# CSE2530 Computational Intelligence
## Assignment 1: Ant Colony Optimization and Genetic Algorithms

 <div>

_Fill in your group number **from Brightspace**, names, and student numbers._
    
| Group          | 71       |
|----------------|---------|
| Lauren But     | 6241468 |
| Bente van Geen | XXXXXXX |
| Femke Knibbe   | XXXXXXX |
| Henno Kruis     | 5988063 |

#### Imports

In [46]:
"""
You may only use numpy to implement your algorithms
You can make use of any other libraries for miscellaneous functions, e.g. to create the visual aids.
Put all of your imports in this code block.
"""
import numpy as np
import numpy.random as random
import sys
import time
import matplotlib.pyplot as plt

"""
The following classes are fully implemented in their own files and you should not change them.
Nonetheless, we encourage you to check how they work; this will help you get started.
"""
from Coordinate import Coordinate
from Direction import Direction
from PathSpecification import PathSpecification
from Route import Route
from SurroundingPheromone import SurroundingPheromone
from TSPData import TSPData


## Part 1: The Travelling Robot Problem
### 1.1 Problem Analysis
#### Question 1:

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 2

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 3

<div style="background-color:#f1be3e">

_Write your answer here._

### 1.2 Genetic Algorithm

In [3]:
# TSP problem solver using genetic algorithms.
class GeneticAlgorithm:

    """
    Constructs a new 'genetic algorithm' object.
    @param generations: the amount of generations.
    @param pop_size: the population size.
    """
    def __init__(self, generations, pop_size):
        self.generations = generations
        self.pop_size = pop_size
    """
    This method should solve the TSP.
    @param tsp_data: the data describing the problem.
    @return the optimized product sequence.
    """
    def solve_tsp(self, tsp_data):
        pass

#### Question 4

<div style="background-color:#f1be3e">

The genes represent the products (locations) that the robot must visit. Each gene corresponds to a single product, identified by its index. Our chromosomes will represent our encoded solution, so in this case the complete route that a robot will follow. We will encode the chromosomes by keeping an ordered sequence of these products, so it is clear in which order the robot visits the products. The chromosome is a permutation of all products, ensuring each product is visited exactly once without duplication.

#### Question 5

<div style="background-color:#f1be3e">

We will use a fitness function based on the total length of the route represented by a chromosome. The total distance for each chromosome is computed by summing the distance from the start position to the first product, the distances between consecutive products and the distance from the last product to the end position. The objective is to minimize this total route length since a shorter route corresponds to a more efficient solution. The fitness function can therefore be defined as the inverse of the route length. This is a suitable choice because shorter routes will result in higher fitness values. This directly reflects the objective of finding the shortest possible route.


#### Question 6

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 7

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 8

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 9

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 10

In [4]:
# Please keep your parameters for the Genetic Algorithm easily changeable here
population_size = 20
generations = 20
persist_file = "./../data/optimal_tsp"

# Setup optimization
tsp_data = TSPData.read_from_file(persist_file)
ga = GeneticAlgorithm(generations, population_size)

# Run optimzation and write to file
solution = ga.solve_tsp(tsp_data)
tsp_data.write_action_file(solution, "./../data/tsp_solution.txt")

TypeError: 'NoneType' object is not subscriptable

<div style="background-color:#f1be3e">

_Put your code extra blocks above (if any) and write your answer here._

## Part 2: Path Finding Through Ant Colony Optimization
### 2.2 Observing the Problem

#### Question 11


- The main purpose for Ant Colony Optimization is to use artificial ants to solve optimization problems, for example a shortest path to a goal problem.

- Ant Colony Optimization is used most often in routing or scheduling problems. For example finding the fastest path from one location to another by car. The main advantage of ACO here is that it can adapt to changes such as busy traffic.

#### Question 12

1. Dead ends
Without a mechanism to recognise and avoid dead ends, ants waste many steps and deposit misleading pheromone that attracts future ants into creating a cycle of ants falling for the same trap.
2. Cycles and loops
Closed loops in the maze allow ants to circle forever without reaching the exit. These are particularly problematic because the pheromone deposited on a cycle reinforces itself over generations, this could cause every ant to enter the loop and the whole generation not finding the exit.
3. Large open spaces
Wide areas with many tiles in all directions, ants lose the directional guidance provided by narrow corridors.
4. Long, winding corridors
Paths that meander far from the optimal direction mislead ants into accumulating pheromone on suboptimal routes. Since these corridors do eventually connect somewhere, they are not penalised naturally by the algorithm, making it hard to distinguish them from shorter alternatives.


#### Question 13

- $\Delta \tau_{ij}^k = \frac{Q}{L_{ij}} $, where $Q$ is a predefined constant and $L_{ij}$ is the length of the link between point $i$ and point $j$, where the points $i$ and $j$ are connected with each other.
- Ants need to drop pheromones in the maze as a type of breadcrumbs for when another ant takes that path. This works because if an ant is on a path with a lot of breadcrumbs it knows that a lot of other ants have been there as well thus it must be a popular path. If an ant finds this type of path, the likelihood of it being an optimal path because of the high frequency of pheromones is high.

#### Question 14

- $\rho \cdot \tau_{ij}$, where $\rho$ is the evaporation constant and $\tau_{ij}$ is the amount of pheromones between points $i$ and $j$.

- Depending on our evaporation constant, every iteration the evaporation constant multiplied with the total amount of pheromone present on the path should be evaporated

- If an ant has walked a path that is not interesting or is not likely to be in the optimal set of paths, we should make sure that this path gets less attractive even though ants might travel along it. Using evaporation makes sure that no chosen path stays attractive and has to be chosen often to stay relevant.

### 2.3 Implementing the Ant Algorithm

In [24]:
# Class that represents the basic Ant functionality
class StandardAnt:

    def __init__(self, maze, path_specification, alpha=1.0, rand=np.random.default_rng()):
        """
        Constructor of a StandardAnt taking a Maze and PathSpecification
        @param maze: the Maze where the ant will try to find a route
        @param path_specification: the PathSpecification consisting of a start and an end coordinate
        """

        self.maze = maze
        self.start: Coordinate = path_specification.get_start()
        self.end: Coordinate = path_specification.get_end()
        self.current_position = self.start
        self.rand = random
        self.alpha = alpha


    def find_route(self,max_steps=10000):
        """
        Method that performs a single complete run through the maze by the ant
        @return the route found by the ant
        """

        route = Route(self.start)
        directions = [e for e in Direction]
        steps = 0

        while not self.current_position.__eq__(self.end):
            if steps > max_steps:
                return None
            surrounding = self.maze.get_surrounding_pheromone(self.current_position)
            pheromones = [surrounding.get(e) ** self.alpha for e in directions]
            total_pheromone = sum(pheromones)
            p_ij = [e / total_pheromone for e in pheromones]

            direction = self.rand.choice(directions, p=p_ij)

            route.add(direction)
            self.current_position = self.current_position.add_direction(direction)
            steps += 1

        return route

In [36]:
# Class that holds all of the maze data.
# This includes the pheromones, the open and blocked tiles in the system,
# and the starting and end coordinates for the ants.
class Maze:

    """
    Constructor of a Maze
    @param walls: array of ints representing the accessible (1) and inaccessible (0) tiles
    @param width: the width (horizontal dimension) of the Maze
    @param length: the length (vertical dimension) of the Maze
    """
    def __init__(self, walls, width, length):
        self.walls = walls
        self.length = length
        self.width = width
        self.start = None
        self.end = None
        self.pheromones = None
        self.initialize_pheromones()

    """
    Initialize pheromones on all tiles of the Maze
    """
    def initialize_pheromones(self):
        self.pheromones = np.copy(np.array(self.walls, dtype=float))

    """
    Reset the Maze for a new shortest path problem
    """
    def reset(self):
        self.initialize_pheromones()

    """
    Update the pheromones along a certain route according to a certain Q
    @param route: the route taken by an ant
    @param q: the normalization factor for the amount of dropped pheromone
    """
    def add_pheromone_route(self, route, q):
        pheromone = (q / route.size())

        current_coordinate = route.get_start()

        if self.in_bounds(current_coordinate):
            self.pheromones[current_coordinate.x][current_coordinate.y] += pheromone
        for d in route.get_route():
            current_coordinate = current_coordinate.add_direction(d)
            if self.in_bounds(current_coordinate):
                self.pheromones[current_coordinate.x][current_coordinate.y] += pheromone

    """
    Update pheromones for a list of routes
    @param routes: a list of routes taken by the ants
    @param q: the normalization factor for the amount of dropped pheromone
    """
    def add_pheromone_routes(self, routes, q):
        for r in routes:
            self.add_pheromone_route(r, q)

    """
    Evaporate pheromone
    @param rho: the evaporation factor
    """
    def evaporate(self, rho):
        self.pheromones *= (1 - rho)

    """
    Getter for the width of the maze
    @return the width of the maze
    """
    def get_width(self):
        return self.width

    """
    Getter for the length of the maze
    @return the length of the maze
    """
    def get_length(self):
        return self.length

    """
    Returns a the amount of pheromones on the neighbouring positions (N/S/E/W)
    @param position: the coordinate where we need to check the surrounding pheromones
    @return the pheromones on the neighbouring coordinates.
    """
    def get_surrounding_pheromone(self, position):
        return SurroundingPheromone(
            self.get_pheromone(position.add_direction(Direction.north)),
            self.get_pheromone(position.add_direction(Direction.east)),
            self.get_pheromone(position.add_direction(Direction.south)),
            self.get_pheromone(position.add_direction(Direction.west))
        )

    """
    Getter for the pheromones on a specific coordinate.
    If the position is not in bounds returns 0
    @param pos: coordinate for the poition of interest
    @return the amount of pheromone at the specified poition
    """
    def get_pheromone(self, pos):
        if not self.in_bounds(pos):
            return 0
        return self.pheromones[pos.x][pos.y]

    """
    Check whether a coordinate lies in the bounds of the current maze
    @param position: the position that we need to check
    @return true if the coordinate lies within the current maze
    """
    def in_bounds(self, position):
        return position.x_between(0, self.width) and position.y_between(0, self.length)

    """
    Representation of Maze as defined by the input file format.
    @return the human-readable representation of a maze
    """
    def __str__(self):
        string = ""
        string += str(self.width)
        string += " "
        string += str(self.length)
        string += " \n"
        for y in range(self.length):
            for x in range(self.width):
                string += str(self.walls[x][y])
                string += " "
            string += "\n"
        return string

    """
    Method that builds a maze from a file
    @param file_path: path to the file which stores the maze
    @return a maze object with pheromones initialized to 0s on inaccessible and 1s on accessible tiles
    """
    @staticmethod
    def create_maze(file_path):
        try:
            f = open(file_path, "r")
            lines = f.read().splitlines()
            dimensions = lines[0].split(" ")
            width = int(dimensions[0])
            length = int(dimensions[1])
            
            #make the maze_layout
            maze_layout = []
            for x in range(width):
                maze_layout.append([])
            
            for y in range(length):
                line = lines[y+1].split(" ")
                for x in range(width):
                    if line[x] != "":
                        state = int(line[x])
                        maze_layout[x].append(state)
            print("Ready reading maze file " + file_path)
            return Maze(maze_layout, width, length)
        except FileNotFoundError:
            print("Error reading maze file " + file_path)

In [43]:
# Class representing the complete ACO algorithm.
# Finds shortest path between two points in a maze according to a path specification.
class AntColonyOptimization:

    def __init__(self, maze, ants_per_gen, generations, q, evaporation, alpha=0.5, alpha_convergence=0.1, beta=2.0, elites=3, random=np.random.default_rng()):
        """
        Constructs a new optimization object using the ant algorithm
        @param maze: the maze (environment) for ants
        @param ants_per_gen: the number of ants per generation (between update of pheromones)
        @param generations: the total number of generations of ants (pheromone updates)
        @param q: the normalization factor for the amount of dropped pheromone
        @param evaporation: the evaporation factor for the pheromones
        @param alpha: the alpha constant for the probability function for the path finding
        @param beta: the beta constant for the probability function for the path finding
        @param random: the random instance of numpy
        """

        self.maze = maze
        self.ants_per_gen = ants_per_gen
        self.generations = generations
        self.q = q
        self.evaporation = evaporation
        self.random = random
        self.alpha = alpha
        self.elites = elites
        self.beta = beta
        self.alpha_convergence = alpha_convergence
        self.ranking = int(ants_per_gen * 0.4)

    def find_shortest_route(self, path_specification):
        """
        Loop that starts the shortest path process
        @param path_specification: description of the route we wish to optimize
        @return the optimized route according to the ACO algorithm
        """

        self.maze.reset()
        shortest_route = None
        alpha = self.alpha
        beta = self.beta
        routes = np.empty(self.ants_per_gen, dtype=Route)

        for i in range(self.generations):
            # print("Generation", i)
            for j in range(self.ants_per_gen):
                ant = IntelligentAnt(self.maze, path_specification, alpha=alpha, beta=beta, rand=self.random)
                routes[j] = ant.find_route()
                if shortest_route is None or routes[j].shorter_than(shortest_route):
                    shortest_route = routes[j]

            percentage_same = sum(1 for occur in routes if occur.size() == routes[0].size())
            if percentage_same >= 0.9 * self.ants_per_gen:
                return shortest_route

            self.maze.evaporate(self.evaporation)
            self.maze.add_pheromone_routes(sorted(routes, key=lambda route: route.size())[:self.ranking], self.q)
            self.maze.add_pheromone_routes([shortest_route] * self.elites, self.q)
            alpha = min(alpha * (1 + self.alpha_convergence), 3.0)
            beta = max(beta * 0.9, 1.0)

        return shortest_route

In [45]:
# Please keep your parameters for the ACO easily changeable here
gen = 10
no_gen = 40
q = 1300
evap = 0.1
alpha = 1.0
beta = 3.0
elites = 3
alpha_convergence = 0.1

# Construct the optimization objects
maze = Maze.create_maze("./../data/hard_maze.txt")
spec = PathSpecification.read_coordinates("./../data/hard_coordinates.txt")
aco = AntColonyOptimization(maze, gen, no_gen, q, evap, alpha=alpha, beta=beta, elites=elites, alpha_convergence=alpha_convergence)

start_time = int(round(time.time() * 1000))
shortest_route = aco.find_shortest_route(spec)

print("Time taken: " + str((int(round(time.time() * 1000)) - start_time) / 1000.0))
print("Route size: " + str(shortest_route.size()))

shortest_route.write_to_file("./../data/hard_solution.txt")

Ready reading maze file ./../data/hard_maze.txt
Time taken: 181.806
Route size: 1393


### 2.4 Upgrading Your Ants with Intelligence

#### Question 15

In [31]:
def get_opposite_direction(dir):
    """
    Get the opposite direction of the current one
    @param dir: direction to take the opposite for
    """
    if dir == Direction.north:
        return Direction.south
    elif dir == Direction.east:
        return Direction.west
    elif dir == Direction.west:
        return Direction.east
    elif dir == Direction.south:
        return Direction.north
    else:
        return None

In [32]:
class AntMemory:
    def __init__(self, initial_pheromones):
        """
        Constructs a new AntMemory object with initial pheromone levels
        @param initial_pheromones: the initial pheromone levels
        """
        self.open_ends = 4
        self.directions = [2, 2, 2, 2]
        self.visited = 1

        for i in range(len(initial_pheromones)):
            if initial_pheromones[i] == 0:
                self.open_ends -= 1
                self.directions[i] = 0

    def visit(self):
        """
        Marks the location as visited
        """
        self.visited *= 2

    def mark_dead_end(self, location):
        """
        Marks a location as a dead end
        @param location: the location to mark, given as a direction from the current position
        """
        if self.directions[Direction.dir_to_int(location)] == 2:
            self.open_ends -= 1
            self.directions[Direction.dir_to_int(location)] = 1

    def get_open_ends(self):
        """
        Returns the number of open paths
        @return: number of open ends from current location
        """
        return self.open_ends

    def is_dead_end(self):
        """
        Checks if the current location is a dead end
        @return: true if the current location is a dead end, false otherwise
        """
        return self.get_open_ends() <= 1

    def __str__(self):
        """
        Returns a string representation of the AntMemory object
        @return: string displaying open ends, visited locations and the state of the neighbouring locations
        """
        return "Open Ends: " + self.open_ends.__str__() + ", Visited: " + self.visited.__str__() + ", Directions: " + self.directions.__str__()

In [33]:
class AntBrain:

    def __init__(self):
        """
        Constructs a new AntBrain object
        """
        self.memory: dict[tuple[int, int], AntMemory] = {}
        self.path = []

    def init_memory(self, pos, memory):
        """
        Initializes memory for a given position if it is not already present
        and marks the position as visited and adds it to the path
        @param pos: the position to initialize memory for
        @param memory: the memory object for the given position
        @return: the updated memory object with the position
        """
        if self.get(pos) is None:
            self.set(pos, memory)
        memory = self.memory[(pos.x, pos.y)]
        self.add_to_path(pos)
        memory.visit()
        return memory

    def set(self, pos, memory):
        """
        Stores and AntMemory object at the position
        @param pos: the position to store memory
        @param memory: the memory to store
        """
        self.memory[(pos.x, pos.y)] = memory

    def get(self, pos, default = None):
        """
        Retrieves the memory at a position, and a default value if the position is not valid
        @param pos: the position to look up
        @param default: the default memory if position is invalid
        return: the memory at position, or the default memory
        """
        if not self.contains(pos):
            return default

        return self.memory.get((pos.x, pos.y), default)

    def contains(self, pos):
        """
        Check if a position is in memory.
        @param pos: the position to check
        @return: true if the position is in memory, false otherwise
        """
        return (pos.x, pos.y) in self.memory

    def is_dead_cycle(self, pos):
        """
        Determines if the given position is part of a dead cycle
        @param pos: the position to check
        @return: true if pos is part of a dead cycle, false otherwise
        """
        if not self.contains(pos):
            return False

        for i in range(len(self.path) - 1, -1, -1):
            if self.path[i].__eq__(pos):
                return True
            if not self.in_dead_cycle(self.path[i]):
                return False

        return False

    def in_dead_cycle(self, pos):
        """
        Check if a position is in a dead cycle based on open ends
        @param pos: the position to check
        @return: true if pos is in a dead cycle, false otherwise
        """
        if self.get(pos) is None:
            return False
        return self.get(pos).get_open_ends() <= 2

    def is_dead_end(self, pos):
        """
        Check if a position is a dead end
        @param pos: the position to check
        @return: true if pos is a dead end, false otherwise
        """
        if self.get(pos) is None:
            return False
        return self.get(pos).is_dead_end()

    def mark_dead_end(self, pos, to_direction):
        """
        Marks a direction from a position as a dead end
        @param pos: the position to mark from
        @param to_direction: the direction in which to mark the dead end
        """
        if self.get(pos) is None:
            return
        self.get(pos).mark_dead_end(to_direction)

        if not self.get(pos).is_dead_end():
            return

        for direction in Direction:
            neighbor_memory = self.get(pos.add_direction(direction))
            if neighbor_memory:
                neighbor_memory.mark_dead_end(get_opposite_direction(direction))

    def remove_from_path(self):
        """
        Removes the last position from the path, if the path is not empty
        """
        if len(self.path) > 0:
            self.path.pop()

    def add_to_path(self, pos: Coordinate):
        """
        Adds a position to the path
        @param pos: the position to add
        """
        if len(self.path) + 1 > 20:
            self.path.pop(0)
        self.path.append(pos)

In [34]:
class IntelligentAnt:
    def __init__(self, maze: Maze, path_specification: PathSpecification, alpha=1.0, beta=3.0, penalty_cycle=5.0, penalty_dead=0.9,
                 rand=np.random.default_rng()):
        """
        Constructor of an IntelligentAnt taking a Maze and PathSpecification
        @param maze: the Maze where the ant will try to find a route
        @param path_specification: the PathSpecification consisting of a start and an end coordinate
        """
        self.maze = maze
        self.start = path_specification.get_start()
        self.end = path_specification.get_end()
        self.current_position: Coordinate = path_specification.get_start()
        self.random = random
        self.beta = beta
        self.alpha = alpha
        self.penalty_cycle = penalty_cycle
        self.penalty_dead = penalty_dead

    def find_route(self):
        """
        Method that performs a single complete run through the maze by the ant
        @return the route found by the ant
        """
        route = Route(self.start)
        brain = AntBrain()

        while not self.current_position.__eq__(self.end):
            memory, pheromones = self.collect_data(brain)

            # Prevent from walking in dead ends
            if memory.is_dead_end() and route.size() > 0:
                memory, pheromones = self.handle_dead_end(brain, memory, route)

            if not memory.is_dead_end() and route.size() > 0:
                pheromones = self.adjust_for_last_direction(pheromones, route)

            pheromones = self.handle_cycles(pheromones, brain, route)

            if memory.is_dead_end() and route.size() > 0:
                memory, pheromones = self.handle_dead_end(brain, memory, route)

            if not memory.is_dead_end() and route.size() > 0:
                pheromones = self.adjust_for_last_direction(pheromones, route)

            self.update_pheromones(brain, pheromones)

            # Update the pheromones with
            direction = self.choose_next_direction(pheromones, route)

            route.add(direction)
            self.current_position = self.current_position.add_direction(direction)

        return route

    def collect_data(self, brain):
        """
        Collects data about the current position's surrounding pheromones and initializes memory
        @param brain: AntBrain object that holds the memory
        @return: the memory and pheromones from the current position and the AntBrain
        """
        surrounding = self.maze.get_surrounding_pheromone(self.current_position)
        pheromones = [surrounding.get(e) for e in Direction]
        memory = brain.init_memory(self.current_position, AntMemory(pheromones))
        return memory, pheromones

    def handle_dead_end(self, brain, memory, route):
        """
        Handles a dead end by backtracking, updating memory
        @param brain: the AntBrain object
        @param memory: the current memory
        @param route: the route on which we can go back
        @return: the updated memory and pheromones
        """
        last_direction, memory = self.walk_dead_ends(route, brain, memory)
        surrounding = self.maze.get_surrounding_pheromone(self.current_position)
        pheromones = [surrounding.get(direction) for direction in Direction]
        pheromones[Direction.dir_to_int(last_direction)] = 0
        return memory, pheromones

    def adjust_for_last_direction(self, pheromones, route):
        """
        Adjust pheromone levels so the ant does not walk back to the same route and slightly prefers walking straight ahead
        @param pheromones: pheromones to adjust
        @param route: route on which to adjust pheromones
        @return: the adjusted pheromones
        """
        last_direction = route.get_route()[-1]
        # Prevent from walking forward and backwards in a loop
        pheromones[Direction.dir_to_int(get_opposite_direction(last_direction))] = 0
        # Make walking forward more attractive
        pheromones[Direction.dir_to_int(last_direction)] *= 1.3
        return pheromones

    def update_pheromones(self, brain, pheromones):
        """
        Update the pheromone levels around the current position
        @param brain: the AntBrain object
        @param pheromones: the pheromones to adjust
        """
        for direction in Direction:
            new_coordinate = self.current_position.add_direction(direction)
            new_memory = brain.get(new_coordinate)

            if new_memory and new_memory.is_dead_end():
                pheromones[Direction.dir_to_int(direction)] = 0

            pheromones[Direction.dir_to_int(direction)] *= self.get_cycle_penalty(brain, new_coordinate)

    def get_cycle_penalty(self, brain, coordinate):
        """
        Calculates the penalty for walking in a cycle
        @param brain: the AntBrain object
        @param coordinate: the coordinate to check for a cycle
        @return: the penalty to apply
        """
        if not brain.contains(coordinate):
            return 1
        return (1 / brain.get(coordinate).visited) ** (self.penalty_cycle)

    def handle_cycles(self, pheromones, brain, route):
        """
        Handle cycling paths by adjusting pheromone levels
        @param pheromones: phermones to adjust
        @param brain: the AntBrain object
        @param route: the route the ant has taken
        @return: the updated pheromones
        """
        for direction in Direction:
            # Skip the pheromones that are 0, since these can never be in a dead cycle
            if pheromones[Direction.dir_to_int(direction)] == 0:
                continue
            directional_coordinate = self.current_position.add_direction(direction)

            if brain.is_dead_cycle(directional_coordinate):
                brain.mark_dead_end(self.current_position, direction)
                pheromones[Direction.dir_to_int(direction)] = 0

        return pheromones


    def choose_next_direction(self, pheromones, route):
        """
        Choose the next direction based on the pheromone probabilities
        @param pheromones: the pheromones of the current position
        @param route: the route the ant has taken
        """
        p_pheromones = [(pheromone ** self.alpha) * ((1 / (route.size() + 1)) ** self.beta) for pheromone in pheromones]
        total_pheromone = sum(p_pheromones)
        if total_pheromone == 0:
            probabilities = 4 / sum(1 for pheromone in pheromones if pheromone > 0)
            p_ij = [probabilities / 4 if pheromone > 0 else 0 for pheromone in pheromones]
            return self.random.choice(list(Direction), p=p_ij)
        p_ij = [p / total_pheromone for p in p_pheromones]
        return self.random.choice(list(Direction), p=p_ij)


    def walk_dead_ends(self, route: Route, brain: AntBrain, memory: AntMemory):
        """
        @param route: the route the ant has taken
        @param brain: the AntBrain object
        @param memory: the memory of the current position
        @return: the last direction before the dead end and the updated memory of the position
        """
        last_direction = None

        while route.size() > 0 and memory.is_dead_end():
            # Remove the dead ends from the route
            last_direction = route.remove_last()
            brain.remove_from_path()

            # Set the current position to the previous position
            brain.mark_dead_end(self.current_position, get_opposite_direction(last_direction))
            self.current_position = self.current_position.subtract_direction(last_direction)
            memory = brain.get(self.current_position)

        return last_direction, memory



Improvements


1. Back-and-forth movement
Ants repeatedly moving between two adjacent squares. Since the pheromone level of the previous square is always at least as high as any unvisited neighbour, the standard algorithm provides no mechanism to prevent this. We resolved this by setting the attractiveness of the direction opposite to the ant's last move to zero, ensuring ants always make forward progress.

2. Memory
Ants were circling around the optimal path without converging onto it. To address this, we equipped ants with a short-term memory storing the last *n* visited coordinates. Any direction leading to a coordinate already in memory is penalised, making revisits less attractive. This improvement had little effect on runtime but improved solution quality, as ants converged to shorter routes using fewer generations.

3. Straight-line preference in open spaces
In large open areas of the maze, often ants lost their way and did not find an exit. We addressed this by creating a mild straight-line preference. This significantly reduced the number of steps ants took to traverse open regions.

4. Dead ends
Finally, ants frequently entered dead ends, wasting steps and depositing misleading pheromone. We tackled this by having ants deposit a near-zero amount of pheromone when backtracking out of a dead end, effectively communicating to future ants that the path is unproductive. This indirectly steers the colony away from dead ends over successive generations and produced a substantial improvement in overall route quality.

### 2.5 Parameter Optimization

#### Question 16

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

#### Question 17

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

### 2.6 The Final Route

#### Question 18

<div style="background-color:#f1be3e">

_Put your code blocks above (if any) and write your answer here._

### 2.7 Synthesis

#### Question 19

In [0]:
# Please keep your parameters for the synthesis part easily changeable here
gen = 1
no_gen = 1
q = 1000
evap = 0.1

persist_file = "./../tmp/my_tsp"
tsp_path = "./../data/tsp_products.txt"
coordinates = "./../data/hard_coordinates.txt"

# Construct optimization
maze = Maze.create_maze("./../data/hard_maze.txt")
tsp_data = TSPData.read_specification(coordinates, tsp_path)
aco = AntColonyOptimization(maze, gen, no_gen, q, evap)

# Run optimization and write to file
tsp_data.calculate_routes(aco)
tsp_data.write_to_file(persist_file)

# Read from file and print
tsp_data2 = TSPData.read_from_file(persist_file)
print(tsp_data == tsp_data2)

# Solve TSP using your own paths file
ga = GeneticAlgorithm(generations, population_size)
solution = ga.solve_tsp(tsp_data2)
tsp_data2.write_action_file(solution, "./../data/tsp_solution.txt")

<div style="background-color:#f1be3e">

_Put your extra code blocks above (if any) and write your answer here._

## Part 3: Open Questions
### 3.1 Reflection

#### Question 20

<div style="background-color:#f1be3e">

_Write your answer here._

#### Question 21

<div style="background-color:#f1be3e">

_Write your answer here._

### 3.2 Pen and Paper

#### Question 22

<div style="background-color:#f1be3e">

_Write your answer here. You can also choose to simply include a photo of your solution._

### 3.3 Division of Work

#### Question 23

<div style="background-color:#f1be3e">


|          Component          |  Name A   |  Name B   |  Name C   |  Name D   |
|-----------------------------|-----------|-----------|-----------|-----------|
| Code (design)               |     A     |     B     |     C     |     D     |
| Code (implementation)       |     A     |     B     |     C     |     D     |
| Code (validation)           |     A     |     B     |     C     |     D     |
| Experiments (execution)     |     A     |     B     |     C     |     D     |
| Experiments (analysis)      |     A     |     B     |     C     |     D     |
| Experiments (visualization) |     A     |     B     |     C     |     D     |
| Report (original draft)     |     A     |     B     |     C     |     D     |
| Report (reviewing, editing) |     A     |     B     |     C     |     D     |

### References

<div style="background-color:#f1be3e">

**If you made use of any non-course resources, cite them below.**